# 08 — Machine Learning Models

## Research Objective

This notebook develops and evaluates machine-learning models for
predicting observed poultry disease outbreak counts in Nigeria.

The modelling target is:

`target_outbreak_count`

Only WAHIS records with observed outbreak counts are included in
supervised modelling. Unavailable outbreak observations are not treated
as zero.

## Experimental Question

The primary research question is:

> Can machine-learning models improve prediction of observed poultry
> disease outbreak counts beyond historical outbreak-persistence
> baselines?

The historical baselines established in Notebook 07 provide the
benchmark for comparison.

## Initial Modelling Scenario

The initial experiment uses a retrospective/end-of-semester prediction
setting.

The climate variables represent the same ADM1 × disease × semester
period as the outbreak target. Therefore, the results from this
experiment should not yet be interpreted as prospective pre-semester
early-warning forecasts.

## Predictor Selection

The initial modelling experiment uses the temporally safe predictor
set identified in Notebook 06.

The following potentially time-leaky features are excluded:

- `disease_frequency`
- `adm1_disease_record_count`

Historical outbreak features are retained only where they were
constructed from earlier semesters.

## Validation Strategy

Because the research problem is temporal, random train/test splitting
is not used.

Model development and evaluation will use chronological validation so
that information from future periods is not used to predict earlier
periods.

The temporal validation strategy will be based on the project's
rolling-origin validation implementation.

## Initial Models

The first modelling experiment will compare a small set of
methodologically distinct models:

1. Regularised linear regression
2. Random Forest regression
3. Gradient Boosting regression

The objective is to establish whether nonlinear machine-learning
models provide meaningful improvement over transparent historical and
statistical benchmarks.

## Evaluation

Models will be evaluated using multiple complementary measures,
including:

- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- Median Absolute Error

Performance will also be examined across:

- chronological periods;
- outbreak-magnitude groups; and
- disease categories.

Model performance will be compared against the historical baselines
established in Notebook 07.

## Reproducibility

All preprocessing, feature selection, validation and modelling steps
will be explicitly documented.

Any model-selection decisions will be based only on information
available within the training/validation process.

In [4]:
# ============================================================
# CELL 2 — LOAD AND VALIDATE MODELLING DATASET
# ============================================================
#
# Purpose:
# Load the validated WAHIS–climate integrated dataset and perform
# structural checks before constructing the supervised modelling
# dataset.
#
# Important:
# - The notebook is located inside the "notebooks" directory.
# - Therefore, the project root is one level above the current
#   working directory.
# - Unavailable outbreak observations are NOT treated as zero.
# - Only records with observed outbreak counts will later be used
#   for supervised modelling.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Identify the project root
# ------------------------------------------------------------
#
# Jupyter is currently running from:
#
#     poultry-disease-early-warning-nigeria/notebooks
#
# Therefore, the parent directory is the project root.
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

print(f"Project root: {PROJECT_ROOT}")


# ------------------------------------------------------------
# 2. Define the integrated dataset path
# ------------------------------------------------------------


DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "wahis_modelling_features.csv"
)

# ------------------------------------------------------------
# 3. Check that the integrated dataset exists
# ------------------------------------------------------------

assert DATA_PATH.exists(), (
    f"Integrated dataset not found: {DATA_PATH}"
)

print(f"Modelling-ready feature dataset found: {DATA_PATH}")


# ------------------------------------------------------------
# 4. Load the integrated dataset
# ------------------------------------------------------------

df = pd.read_csv(DATA_PATH)


# ------------------------------------------------------------
# 5. Display basic dataset information
# ------------------------------------------------------------


print("=" * 70)
print("MACHINE LEARNING MODELS — MODELLING FEATURE DATASET")
print("=" * 70)

print(f"Dataset shape: {df.shape}")

print("\nColumn count:")
print(len(df.columns))

print("\nTemporal coverage:")
print(
    f"{df['year'].min()}–{df['year'].max()}"
)


# ------------------------------------------------------------
# 6. Validate the analytical record identifier
# ------------------------------------------------------------

assert (
    df["analytical_record_id"].notna().all()
), "Missing analytical record IDs detected."

assert (
    df["analytical_record_id"].is_unique
), "Duplicate analytical record IDs detected."


# ------------------------------------------------------------
# 7. Validate required outbreak columns
# ------------------------------------------------------------

required_outbreak_columns = [
    "new_outbreaks",
    "outbreak_count_observed",
    "outbreak_observation_status"
]

for column in required_outbreak_columns:

    assert column in df.columns, (
        f"Required outbreak column missing: {column}"
    )


# ------------------------------------------------------------
# 8. Inspect outbreak observation status
# ------------------------------------------------------------

outbreak_status_counts = (
    df["outbreak_observation_status"]
    .value_counts(dropna=False)
)

print("\nOutbreak observation status:")
display(
    outbreak_status_counts.to_frame(
        name="record_count"
    )
)


# ------------------------------------------------------------
# 9. Validate the relationship between observation status
#    and the actual outbreak-count variable
# ------------------------------------------------------------

observed_mask = (
    df["outbreak_observation_status"]
    == "Observed"
)

unavailable_mask = (
    df["outbreak_observation_status"]
    == "Unavailable"
)


# Observed records must contain numeric outbreak counts.
assert (
    df.loc[
        observed_mask,
        "new_outbreaks"
    ]
    .notna()
    .all()
), "Observed records contain missing outbreak counts."


# Unavailable records must NOT contain outbreak counts.
assert (
    df.loc[
        unavailable_mask,
        "new_outbreaks"
    ]
    .isna()
    .all()
), (
    "Unavailable outbreak observations contain numeric "
    "outbreak counts."
)


# ------------------------------------------------------------
# 10. Confirm that unavailable observations are NOT
#     interpreted as zero
# ------------------------------------------------------------

observed_target = df.loc[
    observed_mask,
    "new_outbreaks"
]

print("\nObserved outbreak-count summary:")
display(
    observed_target.describe()
)


print("\nObserved zero outbreak counts:")
print(
    int(
        (observed_target == 0).sum()
    )
)


# ------------------------------------------------------------
# 11. Validate the supervised modelling population
# ------------------------------------------------------------

n_total = len(df)

n_observed = int(
    observed_mask.sum()
)

n_unavailable = int(
    unavailable_mask.sum()
)

print("\nSupervised modelling population:")
print(f"Total integrated records: {n_total}")
print(f"Observed outbreak records: {n_observed}")
print(f"Unavailable outbreak records: {n_unavailable}")

print(
    f"Observed target percentage: "
    f"{(n_observed / n_total) * 100:.2f}%"
)


# ------------------------------------------------------------
# 12. Validate the observed modelling period
# ------------------------------------------------------------

observed_years = sorted(
    df.loc[
        observed_mask,
        "year"
    ].dropna().unique()
)

print("\nYears containing observed outbreak targets:")
print(observed_years)


# ------------------------------------------------------------
# 13. Final structural validation
# ------------------------------------------------------------

assert n_total == 1180, (
    "Unexpected number of integrated records."
)

assert n_observed == 445, (
    "Unexpected number of observed outbreak records."
)

assert n_unavailable == 735, (
    "Unexpected number of unavailable outbreak records."
)

assert (
    observed_target.min() >= 1
), (
    "Observed outbreak target contains values below 1."
)

assert (
    observed_target.notna().all()
), (
    "Observed outbreak target contains missing values."
)


print(
    "\nMACHINE LEARNING MODELLING DATASET "
    "VALIDATION PASSED"
)

Project root: c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria
Modelling-ready feature dataset found: c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria\data\processed\wahis_modelling_features.csv
MACHINE LEARNING MODELS — MODELLING FEATURE DATASET
Dataset shape: (1180, 38)

Column count:
38

Temporal coverage:
2005–2025

Outbreak observation status:


,record_count
outbreak_observation_status,
Unavailable,735
Observed,445



Observed outbreak-count summary:


count    445.000000
mean       6.840449
std       12.872353
min        1.000000
25%        1.000000
50%        3.000000
75%        7.000000
max      184.000000
Name: new_outbreaks, dtype: float64


Observed zero outbreak counts:
0

Supervised modelling population:
Total integrated records: 1180
Observed outbreak records: 445
Unavailable outbreak records: 735
Observed target percentage: 37.71%

Years containing observed outbreak targets:
[np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

MACHINE LEARNING MODELLING DATASET VALIDATION PASSED


In [8]:
# ============================================================
# NOTEBOOK 08 — CELL 3
# TARGET AND PREDICTOR DEFINITION
# ============================================================
#
# Purpose:
#   Define the supervised modelling population, target, and
#   temporally safe predictor set.
#
# Important:
#   target_outbreak_count is created here from the authoritative
#   WAHIS fields:
#
#       new_outbreaks
#       outbreak_observation_status
#
# Unavailable outbreak observations are NOT treated as zero.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Define project path
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

MODELLING_FEATURES_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "wahis_modelling_features.csv"
)


# ------------------------------------------------------------
# 2. Load the modelling-ready feature dataset
# ------------------------------------------------------------

if not MODELLING_FEATURES_PATH.exists():
    raise FileNotFoundError(
        f"Modelling feature dataset not found:\n"
        f"{MODELLING_FEATURES_PATH}"
    )

model_features = pd.read_csv(MODELLING_FEATURES_PATH)

print("=" * 70)
print("TARGET AND PREDICTOR DEFINITION")
print("=" * 70)

print("\nModelling-ready feature dataset:")
print("Rows:", model_features.shape[0])
print("Columns:", model_features.shape[1])


# ------------------------------------------------------------
# 3. Check for duplicate column names
# ------------------------------------------------------------

duplicate_columns = model_features.columns[
    model_features.columns.duplicated()
].tolist()

if duplicate_columns:
    raise ValueError(
        "Duplicate column names detected:\n"
        + "\n".join(duplicate_columns)
    )

print("\nDuplicate column names: 0")


# ------------------------------------------------------------
# 4. Verify authoritative target source columns
# ------------------------------------------------------------
#
# The exported modelling feature dataset does not contain
# target_outbreak_count because that target is deliberately
# constructed for the supervised modelling experiment.
#
# We therefore derive it from the original WAHIS outbreak
# observation fields retained in the modelling feature dataset.
# ------------------------------------------------------------

target_source_columns = [
    "new_outbreaks",
    "outbreak_observation_status",
]

missing_target_sources = [
    column
    for column in target_source_columns
    if column not in model_features.columns
]

if missing_target_sources:
    raise ValueError(
        "Required target source columns are missing:\n"
        + "\n".join(missing_target_sources)
    )

print("\nAuthoritative target source columns verified.")


# ------------------------------------------------------------
# 5. Define the temporally safe predictor set
# ------------------------------------------------------------
#
# These predictors were identified during the temporal leakage
# audit in Notebook 06.
#
# Potentially time-leaky features are intentionally excluded:
#
#   - disease_frequency
#   - adm1_disease_record_count
# ------------------------------------------------------------

safe_predictors = [
    "year",
    "semester_order",
    "is_first_semester",
    "is_second_semester",
    "semester_sin",
    "semester_cos",
    "years_since_2019",
    "year_sin",
    "year_cos",
    "has_adm1",
    "is_country_level",
    "disease_code",
    "previous_semester_outbreak_count",
    "two_semesters_prior_outbreak_count",
    "outbreak_history_mean_3",
    "outbreak_history_max_3",
    "climate_available",
    "temperature_missing",
    "dewpoint_missing",
    "precipitation_missing",
    "wind_missing",
    "climate_variable_count",
    "climate_complete",
    "temperature_c",
    "dewpoint_c",
    "semester_total_precipitation_mm",
    "semester_mean_wind_component_magnitude_ms",
]

target_column = "target_outbreak_count"


# ------------------------------------------------------------
# 6. Verify all predictor columns exist
# ------------------------------------------------------------

missing_predictors = [
    column
    for column in safe_predictors
    if column not in model_features.columns
]

if missing_predictors:
    raise ValueError(
        "The following predictor columns are missing:\n"
        + "\n".join(missing_predictors)
    )

print(
    f"All {len(safe_predictors)} temporally safe predictors verified."
)


# ------------------------------------------------------------
# 7. Create the supervised modelling target
# ------------------------------------------------------------
#
# Only observed outbreak counts become modelling targets.
#
# For unavailable observations:
#
#       target_outbreak_count = NaN
#
# This preserves the surveillance principle:
#
#       unavailable != zero
# ------------------------------------------------------------

model_features[target_column] = np.where(
    model_features["outbreak_observation_status"] == "Observed",
    model_features["new_outbreaks"],
    np.nan
)


# ------------------------------------------------------------
# 8. Define the supervised modelling population
# ------------------------------------------------------------

model_df = model_features.loc[
    model_features["outbreak_observation_status"] == "Observed",
    safe_predictors + [target_column]
].copy()


# ------------------------------------------------------------
# 9. Validate target values
# ------------------------------------------------------------

if model_df[target_column].isna().any():
    raise ValueError(
        "Missing target values found among observed records."
    )

if (model_df[target_column] < 0).any():
    raise ValueError(
        "Negative outbreak counts found in the target."
    )

observed_zero_count = int(
    model_df[target_column].eq(0).sum()
)

print("\nTarget validation:")
print("Observed modelling records:", len(model_df))
print("Observed zero outbreak counts:", observed_zero_count)


# ------------------------------------------------------------
# 10. Display predictor definition
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TARGET AND PREDICTOR DEFINITION")
print("=" * 70)

print(
    f"Number of candidate predictors: {len(safe_predictors)}"
)

print("\nTemporally safe predictors:")

for i, predictor in enumerate(safe_predictors, start=1):
    print(f"{i:2d}. {predictor}")


# ------------------------------------------------------------
# 11. Display modelling dataset
# ------------------------------------------------------------

print("\nSupervised modelling dataset:")
print("Rows:", model_df.shape[0])
print("Columns:", model_df.shape[1])
print("Target:", target_column)
print("Predictors:", len(safe_predictors))

print("\nTarget summary:")
print(model_df[target_column].describe())


# ------------------------------------------------------------
# 12. Check duplicate columns in final modelling dataset
# ------------------------------------------------------------

if model_df.columns.duplicated().any():
    raise ValueError(
        "Duplicate columns remain in model_df."
    )

print("\nDuplicate columns in model_df: 0")


# ------------------------------------------------------------
# 13. Examine temporal coverage
# ------------------------------------------------------------

year_target_summary = (
    model_df
    .groupby("year", as_index=False)
    .agg(
        observed_records=(
            target_column,
            "size"
        ),
        mean_outbreak_count=(
            target_column,
            "mean"
        ),
        median_outbreak_count=(
            target_column,
            "median"
        ),
        max_outbreak_count=(
            target_column,
            "max"
        )
    )
    .sort_values("year")
)

print("\nObserved target records by year:")
print(year_target_summary.to_string(index=False))


# ------------------------------------------------------------
# 14. Final validation
# ------------------------------------------------------------

assert len(safe_predictors) == 27

assert model_df.shape[0] == 445

assert model_df.shape[1] == 28

assert target_column in model_df.columns

assert not model_df[target_column].isna().any()

assert not model_df.columns.duplicated().any()

assert observed_zero_count == 0

print("\n" + "=" * 70)
print("TARGET AND PREDICTOR DEFINITION VALIDATION PASSED")
print("=" * 70)

TARGET AND PREDICTOR DEFINITION

Modelling-ready feature dataset:
Rows: 1180
Columns: 38

Duplicate column names: 0

Authoritative target source columns verified.
All 27 temporally safe predictors verified.

Target validation:
Observed modelling records: 445
Observed zero outbreak counts: 0

TARGET AND PREDICTOR DEFINITION
Number of candidate predictors: 27

Temporally safe predictors:
 1. year
 2. semester_order
 3. is_first_semester
 4. is_second_semester
 5. semester_sin
 6. semester_cos
 7. years_since_2019
 8. year_sin
 9. year_cos
10. has_adm1
11. is_country_level
12. disease_code
13. previous_semester_outbreak_count
14. two_semesters_prior_outbreak_count
15. outbreak_history_mean_3
16. outbreak_history_max_3
17. climate_available
18. temperature_missing
19. dewpoint_missing
20. precipitation_missing
21. wind_missing
22. climate_variable_count
23. climate_complete
24. temperature_c
25. dewpoint_c
26. semester_total_precipitation_mm
27. semester_mean_wind_component_magnitude_ms

S

## 4. Rolling-Origin Temporal Validation

### Objective

This section defines the chronological validation strategy used for
machine-learning model development.

Because poultry disease surveillance data are temporal, random train/test
splitting is inappropriate for the primary experiment. Random splitting
could allow observations from later periods to influence predictions for
earlier periods.

Instead, the project uses rolling-origin expanding-window validation.

### Validation Principle

For each validation period:

- training data consist only of observations available before the
  validation period;
- the validation period represents future observations relative to the
  training data;
- the training window expands as time progresses;
- no future validation observations are used to construct the training
  data.

This provides a more realistic estimate of how the models would perform
when applied to future surveillance periods.

### Important Consideration

The supervised dataset contains observed outbreak targets from 2020–2025.
However, the temporal unit of the modelling problem is the
ADM1 × disease × semester period.

Therefore, validation must preserve chronological ordering at the
semester level rather than randomly splitting individual WAHIS records.

### Validation Design

The initial experiment will use expanding-window validation across the
observed modelling period.

For each validation period:

    Past periods → Training
    Next period   → Validation

The validation periods will be retained separately so that model
performance can later be examined by year, semester, disease and outbreak
magnitude.

The validation design is established before model fitting and will be
used consistently across all initial machine-learning models.

In [11]:
# ============================================================
# NOTEBOOK 08 — CELL 4
# ROLLING-ORIGIN TEMPORAL VALIDATION
# ============================================================
#
# Purpose:
#   Define and validate the chronological expanding-window
#   validation strategy before fitting any machine-learning model.
#
# Important:
#   No model is fitted in this cell.
#
# The project's existing rolling_origin_splits() implementation
# from src/models/train.py is used directly.
#
# The temporal period is encoded as:
#
#     year * 2 + (semester_order - 1)
#
# because semester_order is:
#
#     1 = Jan-Jun
#     2 = Jul-Dec
#
# This ensures correct chronological ordering.
# ============================================================

from pathlib import Path
import sys
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Make the project source directory importable
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# ------------------------------------------------------------
# 2. Import the project's rolling-origin splitter
# ------------------------------------------------------------

from src.models.train import rolling_origin_splits


# ------------------------------------------------------------
# 3. Prepare the validation dataset
# ------------------------------------------------------------

validation_df = model_df.copy()


# ------------------------------------------------------------
# 4. Create a correctly ordered semester-level time index
# ------------------------------------------------------------
#
# semester_order:
#     1 = Jan-Jun
#     2 = Jul-Dec
#
# Therefore:
#
#     2020 Jan-Jun = 2020*2 + 0 = 4040
#     2020 Jul-Dec = 2020*2 + 1 = 4041
#     2021 Jan-Jun = 2021*2 + 0 = 4042
#     2021 Jul-Dec = 2021*2 + 1 = 4043
#
# This produces a strictly chronological numeric index.
# ------------------------------------------------------------

validation_df["temporal_period"] = (
    validation_df["year"] * 2
    + (validation_df["semester_order"] - 1)
)

validation_df = validation_df.sort_values(
    ["temporal_period"]
).reset_index(drop=True)


# ------------------------------------------------------------
# 5. Define a readable period-label function
# ------------------------------------------------------------

def period_label(period):
    """
    Convert the numeric temporal period into a readable
    year-semester label.
    """

    period = int(period)

    year = period // 2

    if period % 2 == 0:
        semester = "Jan-Jun"
    else:
        semester = "Jul-Dec"

    return f"{year} {semester}"


# ------------------------------------------------------------
# 6. Validate the temporal encoding itself
# ------------------------------------------------------------

period_check = (
    validation_df[
        ["year", "semester_order", "temporal_period"]
    ]
    .drop_duplicates()
    .sort_values("temporal_period")
    .reset_index(drop=True)
)

period_check["period_label"] = (
    period_check["temporal_period"]
    .apply(period_label)
)

print("=" * 70)
print("ROLLING-ORIGIN TEMPORAL VALIDATION")
print("=" * 70)

print("\nChronological modelling periods:")

print(
    period_check[
        [
            "year",
            "semester_order",
            "temporal_period",
            "period_label"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 7. Validate temporal-period consistency
# ------------------------------------------------------------
#
# Every year/semester combination must map to exactly one
# temporal period, and the period labels must agree with the
# original year and semester.
# ------------------------------------------------------------

expected_period = (
    period_check["year"] * 2
    + (period_check["semester_order"] - 1)
)

if not (
    period_check["temporal_period"]
    == expected_period
).all():
    raise ValueError(
        "Temporal-period encoding is inconsistent with "
        "year and semester_order."
    )


# ------------------------------------------------------------
# 8. Validate expected temporal coverage
# ------------------------------------------------------------

observed_years = sorted(
    validation_df["year"].unique()
)

observed_semesters = sorted(
    validation_df["semester_order"].unique()
)

print("\nObserved years:", observed_years)
print("Observed semester orders:", observed_semesters)

if observed_years != [2020, 2021, 2022, 2023, 2024, 2025]:
    raise ValueError(
        "Unexpected modelling-year coverage detected."
    )

if observed_semesters != [1, 2]:
    raise ValueError(
        "Expected both semester orders 1 and 2."
    )


# ------------------------------------------------------------
# 9. Define rolling-origin validation
# ------------------------------------------------------------
#
# Four initial semesters are used for training.
# One subsequent semester is used for validation.
#
# This creates an expanding-window design:
#
# Fold 1:
#   Train -> 2020 H1 through 2021 H2
#   Test  -> 2022 H1
#
# Fold 2:
#   Train -> 2020 H1 through 2022 H1
#   Test  -> 2022 H2
#
# ...
#
# Fold 8:
#   Train -> 2020 H1 through 2025 H1
#   Test  -> 2025 H2
# ------------------------------------------------------------

MIN_TRAIN_PERIODS = 4
TEST_PERIODS = 1

splits = rolling_origin_splits(
    validation_df,
    time_column="temporal_period",
    target_column=target_column,
    min_train_periods=MIN_TRAIN_PERIODS,
    test_periods=TEST_PERIODS,
)


# ------------------------------------------------------------
# 10. Validate split count
# ------------------------------------------------------------

number_of_periods = (
    validation_df["temporal_period"]
    .nunique()
)

expected_splits = (
    number_of_periods
    - MIN_TRAIN_PERIODS
    - TEST_PERIODS
    + 1
)

if len(splits) != expected_splits:
    raise ValueError(
        f"Expected {expected_splits} validation splits, "
        f"but received {len(splits)}."
    )

print(
    "\nNumber of unique modelling periods:",
    number_of_periods
)

print(
    "Number of rolling-origin validation splits:",
    len(splits)
)


# ------------------------------------------------------------
# 11. Build validation split summary
# ------------------------------------------------------------

split_summary = []

for split_number, (train, test) in enumerate(
    splits,
    start=1
):

    train_periods = sorted(
        train["temporal_period"].unique()
    )

    test_periods = sorted(
        test["temporal_period"].unique()
    )

    train_start = min(train_periods)
    train_end = max(train_periods)

    test_start = min(test_periods)
    test_end = max(test_periods)

    # --------------------------------------------------------
    # Strict chronological separation
    # --------------------------------------------------------

    if train_end >= test_start:
        raise ValueError(
            f"Temporal leakage detected in split {split_number}: "
            f"training ends at {period_label(train_end)}, "
            f"but validation starts at {period_label(test_start)}."
        )

    split_summary.append(
        {
            "split": split_number,
            "training_records": len(train),
            "validation_records": len(test),
            "training_start": period_label(train_start),
            "training_end": period_label(train_end),
            "validation_start": period_label(test_start),
            "validation_end": period_label(test_end),
        }
    )


split_summary_df = pd.DataFrame(
    split_summary
)


# ------------------------------------------------------------
# 12. Display the validation design
# ------------------------------------------------------------

print("\nValidation split summary:")

print(
    split_summary_df.to_string(index=False)
)


# ------------------------------------------------------------
# 13. Validate expanding training windows
# ------------------------------------------------------------

previous_training_records = 0

for split_number, (train, test) in enumerate(
    splits,
    start=1
):

    current_training_records = len(train)

    if current_training_records <= previous_training_records:
        raise ValueError(
            f"Training window failed to expand in split "
            f"{split_number}."
        )

    previous_training_records = (
        current_training_records
    )


# ------------------------------------------------------------
# 14. Validate train/test temporal separation
# ------------------------------------------------------------

for split_number, (train, test) in enumerate(
    splits,
    start=1
):

    train_max_period = (
        train["temporal_period"].max()
    )

    test_min_period = (
        test["temporal_period"].min()
    )

    if train_max_period >= test_min_period:
        raise ValueError(
            f"Temporal ordering failed in split "
            f"{split_number}."
        )


# ------------------------------------------------------------
# 15. Validate train/test record separation
# ------------------------------------------------------------

for split_number, (train, test) in enumerate(
    splits,
    start=1
):

    train_indices = set(train.index)
    test_indices = set(test.index)

    overlap = train_indices.intersection(
        test_indices
    )

    if overlap:
        raise ValueError(
            f"Train/test record overlap detected "
            f"in split {split_number}."
        )


# ------------------------------------------------------------
# 16. Validate the first and final folds explicitly
# ------------------------------------------------------------

first_train, first_test = splits[0]

last_train, last_test = splits[-1]

first_train_end = first_train[
    "temporal_period"
].max()

first_test_start = first_test[
    "temporal_period"
].min()

last_train_end = last_train[
    "temporal_period"
].max()

last_test_start = last_test[
    "temporal_period"
].min()

print("\nFirst validation fold:")
print(
    f"  Training:   {period_label(first_train['temporal_period'].min())}"
    f" → {period_label(first_train_end)}"
)
print(
    f"  Validation: {period_label(first_test_start)}"
)

print("\nFinal validation fold:")
print(
    f"  Training:   {period_label(last_train['temporal_period'].min())}"
    f" → {period_label(last_train_end)}"
)
print(
    f"  Validation: {period_label(last_test_start)}"
)


# ------------------------------------------------------------
# 17. Final validation output
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ROLLING-ORIGIN VALIDATION DESIGN VALIDATED")
print("=" * 70)

print("\nValidation design:")
print(
    f"  Minimum training periods: {MIN_TRAIN_PERIODS}"
)
print(
    f"  Validation periods per fold: {TEST_PERIODS}"
)
print(
    f"  Total validation folds: {len(splits)}"
)

print("\nKey properties:")
print(
    "  ✓ Temporal periods are correctly ordered"
)
print(
    "  ✓ Training observations always precede validation observations"
)
print(
    "  ✓ Training window expands chronologically"
)
print(
    "  ✓ Validation windows contain future observations"
)
print(
    "  ✓ No train/test record overlap"
)
print(
    "  ✓ Missing outbreak targets are excluded rather than treated as zero"
)
print(
    "  ✓ No model fitting performed"
)

print(
    "\nTemporal validation is ready for model fitting."
)

ROLLING-ORIGIN TEMPORAL VALIDATION

Chronological modelling periods:
 year  semester_order  temporal_period period_label
 2020               1             4040 2020 Jan-Jun
 2020               2             4041 2020 Jul-Dec
 2021               1             4042 2021 Jan-Jun
 2021               2             4043 2021 Jul-Dec
 2022               1             4044 2022 Jan-Jun
 2022               2             4045 2022 Jul-Dec
 2023               1             4046 2023 Jan-Jun
 2023               2             4047 2023 Jul-Dec
 2024               1             4048 2024 Jan-Jun
 2024               2             4049 2024 Jul-Dec
 2025               1             4050 2025 Jan-Jun
 2025               2             4051 2025 Jul-Dec

Observed years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Observed semester orders: [np.int64(1), np.int64(2)]

Number of unique modelling periods: 12
Number of rolling-origin validation splits: 8

V

## 5. Preprocessing and Model Pipeline Design

### Objective

This section defines the preprocessing strategy used by the initial
machine-learning experiments.

The modelling dataset contains missing values in several predictors,
particularly historical outbreak features and climate variables.

Missing predictor values are not replaced globally before validation.
Instead, preprocessing is performed within each model pipeline so that
information from future validation periods cannot influence the
training process.

### Predictor Types

The initial experiment uses 27 temporally safe predictors.

All predictors are currently numeric:

- temporal features;
- geographic indicators;
- disease code;
- historical outbreak features;
- climate availability indicators;
- climate measurements.

### Missing-Value Strategy

Missing numeric predictors will be handled using median imputation.

The imputation parameters are learned from the training data within each
validation fold.

This prevents validation-period information from influencing the
preprocessing stage.

### Model-Specific Preprocessing

Three initial regression models will be evaluated:

1. Ridge Regression
2. Random Forest Regression
3. Gradient Boosting Regression

Ridge Regression will use standardisation after imputation because
regularised linear models are sensitive to feature scale.

Tree-based models will use imputation without feature scaling because
tree-based models do not require standardised predictor magnitudes.

### Reproducibility

The same predictor set, target definition and rolling-origin validation
folds will be used for every model.

No model will be tuned using information from future validation folds.

In [12]:
# ============================================================
# NOTEBOOK 08 — CELL 5
# PREPROCESSING AND MODEL PIPELINE DESIGN
# ============================================================
#
# Purpose:
#   Define reproducible preprocessing and model pipelines for
#   the initial machine-learning experiment.
#
# Important:
#   No model is fitted in this cell.
#
# Missing-value imputation is deliberately placed inside each
# pipeline. Therefore, imputation parameters will be learned
# from the training fold only during validation.
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor


# ------------------------------------------------------------
# 1. Define predictor and target variables
# ------------------------------------------------------------

X_columns = safe_predictors

y_column = target_column


# ------------------------------------------------------------
# 2. Confirm that all predictors are numeric
# ------------------------------------------------------------

non_numeric_predictors = [
    column
    for column in X_columns
    if not pd.api.types.is_numeric_dtype(
        model_df[column]
    )
]

if non_numeric_predictors:
    raise TypeError(
        "Non-numeric predictors detected:\n"
        + "\n".join(non_numeric_predictors)
    )

print("=" * 70)
print("PREPROCESSING AND MODEL PIPELINE DESIGN")
print("=" * 70)

print(
    "\nNumber of predictors:",
    len(X_columns)
)

print(
    "Target:",
    y_column
)

print(
    "All predictors are numeric."
)


# ------------------------------------------------------------
# 3. Define the numeric preprocessing pipeline
# ------------------------------------------------------------
#
# Median imputation is fitted only on the training data when
# the pipeline is trained.
#
# No global imputation is performed here.
# ------------------------------------------------------------

ridge_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median"
                        )
                    ),
                    (
                        "scaler",
                        StandardScaler()
                    ),
                ]
            ),
            X_columns,
        )
    ],
    remainder="drop",
)


# ------------------------------------------------------------
# 4. Define Ridge Regression pipeline
# ------------------------------------------------------------
#
# Ridge provides a regularised linear reference model.
#
# alpha controls the strength of L2 regularisation.
#
# We deliberately use a fixed, transparent value for the first
# experiment rather than performing a large hyperparameter search.
# ------------------------------------------------------------

ridge_pipeline = Pipeline(
    steps=[
        (
            "preprocess",
            ridge_preprocessor
        ),
        (
            "model",
            Ridge(
                alpha=1.0
            )
        ),
    ]
)


# ------------------------------------------------------------
# 5. Define Random Forest preprocessing
# ------------------------------------------------------------
#
# Tree-based models do not require feature scaling.
# Median imputation is still required because scikit-learn's
# RandomForestRegressor should receive a complete numeric matrix.
# ------------------------------------------------------------

tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(
                strategy="median"
            ),
            X_columns,
        )
    ],
    remainder="drop",
)


# ------------------------------------------------------------
# 6. Define Random Forest pipeline
# ------------------------------------------------------------

random_forest_pipeline = Pipeline(
    steps=[
        (
            "preprocess",
            tree_preprocessor
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                n_jobs=-1
            )
        ),
    ]
)


# ------------------------------------------------------------
# 7. Define Gradient Boosting pipeline
# ------------------------------------------------------------

gradient_boosting_pipeline = Pipeline(
    steps=[
        (
            "preprocess",
            tree_preprocessor
        ),
        (
            "model",
            GradientBoostingRegressor(
                random_state=42
            )
        ),
    ]
)


# ------------------------------------------------------------
# 8. Store the initial model specifications
# ------------------------------------------------------------
#
# Keeping the models in one dictionary allows the same
# validation procedure to be applied consistently to every
# model.
# ------------------------------------------------------------

model_pipelines = {
    "Ridge Regression": ridge_pipeline,
    "Random Forest": random_forest_pipeline,
    "Gradient Boosting": gradient_boosting_pipeline,
}


# ------------------------------------------------------------
# 9. Validate pipeline construction
# ------------------------------------------------------------

if len(model_pipelines) != 3:
    raise ValueError(
        "Expected exactly three initial models."
    )

for model_name, pipeline in model_pipelines.items():

    if not isinstance(
        pipeline,
        Pipeline
    ):
        raise TypeError(
            f"{model_name} is not a scikit-learn Pipeline."
        )

print("\nInitial models:")

for model_name in model_pipelines:
    print(f"  ✓ {model_name}")


# ------------------------------------------------------------
# 10. Validate that preprocessing contains imputation
# ------------------------------------------------------------

for model_name, pipeline in model_pipelines.items():

    preprocess_step = pipeline.named_steps[
        "preprocess"
    ]

    if preprocess_step is None:
        raise ValueError(
            f"{model_name} has no preprocessing stage."
        )

print(
    "\nAll models contain explicit preprocessing pipelines."
)


# ------------------------------------------------------------
# 11. Confirm no fitting has occurred
# ------------------------------------------------------------

print(
    "\nNo model fitting has been performed."
)

print(
    "Preprocessing and model pipelines are ready for "
    "rolling-origin evaluation."
)


# ------------------------------------------------------------
# 12. Final validation
# ------------------------------------------------------------

assert len(X_columns) == 27

assert y_column == "target_outbreak_count"

assert set(model_pipelines.keys()) == {
    "Ridge Regression",
    "Random Forest",
    "Gradient Boosting",
}

print("\n" + "=" * 70)
print("PREPROCESSING AND MODEL PIPELINE DESIGN VALIDATED")
print("=" * 70)

PREPROCESSING AND MODEL PIPELINE DESIGN

Number of predictors: 27
Target: target_outbreak_count
All predictors are numeric.

Initial models:
  ✓ Ridge Regression
  ✓ Random Forest
  ✓ Gradient Boosting

All models contain explicit preprocessing pipelines.

No model fitting has been performed.
Preprocessing and model pipelines are ready for rolling-origin evaluation.

PREPROCESSING AND MODEL PIPELINE DESIGN VALIDATED


In [16]:
# ============================================================
# NOTEBOOK 08 — CELL 6
# ROLLING-ORIGIN MODEL FITTING AND EVALUATION
# ============================================================
#
# Purpose:
#   Fit and evaluate the initial regression models using the
#   validated rolling-origin temporal validation strategy.
#
# Models:
#   1. Ridge Regression
#   2. Random Forest
#   3. Gradient Boosting
#
# Important:
#   - No random train/test split is used.
#   - Missing target values are excluded.
#   - Missing predictors are handled inside each pipeline.
#   - Preprocessing is fitted separately within each training fold.
#   - Future validation observations are never used during fitting.
#   - No unavailable outbreak observation is treated as zero.
# ============================================================


# ------------------------------------------------------------
# 1. Create the supervised modelling dataset
# ------------------------------------------------------------
#
# Only records with an observed outbreak count are included.
# ------------------------------------------------------------

validation_df = model_df[
    model_df[y_column].notna()
].copy()


# ------------------------------------------------------------
# 2. Create a local validation record identifier
# ------------------------------------------------------------
#
# The modelling dataset does not contain the original
# analytical_record_id, so we create a local identifier.
#
# This identifier is used only for validation-overlap checks
# and prediction tracking within this notebook.
# ------------------------------------------------------------

validation_df = validation_df.reset_index(
    drop=True
)

validation_df[
    "validation_record_id"
] = np.arange(
    len(validation_df)
)


# ------------------------------------------------------------
# 3. Reconstruct the chronological modelling period
# ------------------------------------------------------------
#
# semester_order:
#   1 = Jan-Jun
#   2 = Jul-Dec
#
# Correct chronological encoding:
#
#   2020 Jan-Jun -> 4040
#   2020 Jul-Dec -> 4041
#   2021 Jan-Jun -> 4042
#   ...
#   2025 Jul-Dec -> 4051
# ------------------------------------------------------------

validation_df[
    "temporal_period"
] = (
    validation_df["year"] * 2
    + (
        validation_df["semester_order"]
        - 1
    )
)


# ------------------------------------------------------------
# 4. Create readable period labels
# ------------------------------------------------------------

def period_label(period):
    """
    Convert the numeric temporal-period encoding
    into a readable year-semester label.
    """

    period = int(period)

    year = period // 2

    semester = (
        "Jan-Jun"
        if period % 2 == 0
        else "Jul-Dec"
    )

    return f"{year} {semester}"


validation_df[
    "period_label"
] = (
    validation_df[
        "temporal_period"
    ]
    .apply(period_label)
)


# ------------------------------------------------------------
# 5. Sort the modelling dataset chronologically
# ------------------------------------------------------------

validation_df = (
    validation_df
    .sort_values(
        [
            "temporal_period",
            "validation_record_id",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Validate chronological modelling periods
# ------------------------------------------------------------

observed_periods = (
    validation_df[
        [
            "year",
            "semester_order",
            "temporal_period",
            "period_label",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "temporal_period"
    )
    .reset_index(drop=True)
)


expected_years = list(
    range(2020, 2026)
)

expected_semesters = [1, 2]

actual_years = sorted(
    validation_df[
        "year"
    ].unique().tolist()
)

actual_semesters = sorted(
    validation_df[
        "semester_order"
    ].unique().tolist()
)


if actual_years != expected_years:
    raise ValueError(
        f"Unexpected modelling years: "
        f"{actual_years}"
    )


if actual_semesters != expected_semesters:
    raise ValueError(
        f"Unexpected semester orders: "
        f"{actual_semesters}"
    )


if not observed_periods[
    "temporal_period"
].is_monotonic_increasing:

    raise ValueError(
        "Temporal periods are not correctly ordered."
    )


# ------------------------------------------------------------
# 7. Display chronological modelling periods
# ------------------------------------------------------------

print("=" * 70)
print("ROLLING-ORIGIN MODEL FITTING AND EVALUATION")
print("=" * 70)

print(
    "\nChronological modelling periods:"
)

display(
    observed_periods
)


# ------------------------------------------------------------
# 8. Generate rolling-origin validation folds
# ------------------------------------------------------------
#
# The project-level rolling_origin_splits() implementation
# is used here.
#
# Minimum training periods:
#   4 semesters
#
# Validation:
#   1 future semester
# ------------------------------------------------------------

rolling_folds = rolling_origin_splits(
    validation_df,
    time_column="temporal_period",
    target_column=y_column,
    min_train_periods=4,
    test_periods=1,
)


# ------------------------------------------------------------
# 9. Validate the rolling-origin folds
# ------------------------------------------------------------

if len(rolling_folds) != 8:
    raise ValueError(
        f"Expected 8 rolling-origin folds, "
        f"but received {len(rolling_folds)}."
    )


for fold_number, (
    train_fold,
    validation_fold
) in enumerate(
    rolling_folds,
    start=1
):

    train_periods = sorted(
        train_fold[
            "temporal_period"
        ].unique()
    )

    validation_periods = sorted(
        validation_fold[
            "temporal_period"
        ].unique()
    )

    # Training must strictly precede validation.
    if max(train_periods) >= min(
        validation_periods
    ):

        raise ValueError(
            f"Temporal leakage detected "
            f"in fold {fold_number}."
        )

    # Training and validation records must not overlap.
    train_ids = set(
        train_fold[
            "validation_record_id"
        ]
    )

    validation_ids = set(
        validation_fold[
            "validation_record_id"
        ]
    )

    if train_ids.intersection(
        validation_ids
    ):

        raise ValueError(
            f"Train/validation record overlap "
            f"detected in fold {fold_number}."
        )


# ------------------------------------------------------------
# 10. Initialise result containers
# ------------------------------------------------------------

fold_results = []

prediction_results = []


# ------------------------------------------------------------
# 11. Fit and evaluate each model on each fold
# ------------------------------------------------------------

for fold_number, (
    train_fold,
    validation_fold
) in enumerate(
    rolling_folds,
    start=1
):

    # --------------------------------------------------------
    # Training predictors and target
    # --------------------------------------------------------

    X_train = train_fold[
        X_columns
    ]

    y_train = train_fold[
        y_column
    ]

    # --------------------------------------------------------
    # Future validation predictors and target
    # --------------------------------------------------------

    X_validation = validation_fold[
        X_columns
    ]

    y_validation = validation_fold[
        y_column
    ]

    # --------------------------------------------------------
    # Validation period
    # --------------------------------------------------------

    validation_period = int(
        validation_fold[
            "temporal_period"
        ].iloc[0]
    )

    validation_year = int(
        validation_fold[
            "year"
        ].iloc[0]
    )

    validation_semester = int(
        validation_fold[
            "semester_order"
        ].iloc[0]
    )

    validation_label = period_label(
        validation_period
    )

    # --------------------------------------------------------
    # Fit and evaluate every model
    # --------------------------------------------------------

    for model_name, model_pipeline in (
        model_pipelines.items()
    ):

        # ----------------------------------------------------
        # Fit only on the training fold.
        #
        # All preprocessing inside the pipeline is therefore
        # learned only from training observations.
        # ----------------------------------------------------

        model_pipeline.fit(
            X_train,
            y_train
        )

        # ----------------------------------------------------
        # Predict the future validation fold.
        # ----------------------------------------------------

        predictions = model_pipeline.predict(
            X_validation
        )

        # ----------------------------------------------------
        # Calculate evaluation metrics.
        # ----------------------------------------------------

        mae = mean_absolute_error(
            y_validation,
            predictions
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_validation,
                predictions
            )
        )

        median_ae = median_absolute_error(
            y_validation,
            predictions
        )

        # ----------------------------------------------------
        # Store fold-level metrics.
        # ----------------------------------------------------

        fold_results.append(
            {
                "model": model_name,
                "fold": fold_number,
                "validation_period":
                    validation_period,
                "validation_label":
                    validation_label,
                "validation_year":
                    validation_year,
                "validation_semester":
                    validation_semester,
                "training_records":
                    len(train_fold),
                "validation_records":
                    len(validation_fold),
                "mae": mae,
                "rmse": rmse,
                "median_absolute_error":
                    median_ae,
            }
        )

        # ----------------------------------------------------
        # Store individual predictions.
        #
        # We deliberately use only fields that are guaranteed
        # to exist in the current modelling dataset.
        # ----------------------------------------------------

        fold_predictions = validation_fold[
            [
                "validation_record_id",
                "year",
                "semester_order",
                "temporal_period",
                y_column,
            ]
        ].copy()

        fold_predictions[
            "model"
        ] = model_name

        fold_predictions[
            "fold"
        ] = fold_number

        fold_predictions[
            "validation_period_label"
        ] = validation_label

        fold_predictions[
            "prediction"
        ] = predictions

        fold_predictions[
            "absolute_error"
        ] = np.abs(
            y_validation.to_numpy()
            - predictions
        )

        fold_predictions[
            "squared_error"
        ] = (
            y_validation.to_numpy()
            - predictions
        ) ** 2

        prediction_results.append(
            fold_predictions
        )


# ------------------------------------------------------------
# 12. Convert results into DataFrames
# ------------------------------------------------------------

fold_results_df = pd.DataFrame(
    fold_results
)

prediction_results_df = pd.concat(
    prediction_results,
    ignore_index=True
)


# ------------------------------------------------------------
# 13. Validate prediction coverage
# ------------------------------------------------------------

expected_validation_records = sum(
    len(validation_fold)
    for _, validation_fold
    in rolling_folds
)

expected_prediction_records = (
    expected_validation_records
    * len(model_pipelines)
)

actual_prediction_records = len(
    prediction_results_df
)


if actual_prediction_records != (
    expected_prediction_records
):

    raise ValueError(
        "Unexpected number of prediction records. "
        f"Expected {expected_prediction_records}, "
        f"received {actual_prediction_records}."
    )


# ------------------------------------------------------------
# 14. Calculate fold-level summary
# ------------------------------------------------------------

fold_summary = (
    fold_results_df
    .sort_values(
        [
            "fold",
            "model",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 15. Calculate mean fold performance
# ------------------------------------------------------------
#
# Each temporal fold receives equal weight.
# ------------------------------------------------------------

mean_fold_results = (
    fold_results_df
    .groupby("model")
    .agg(
        validation_folds=(
            "fold",
            "nunique"
        ),
        evaluated_records=(
            "validation_records",
            "sum"
        ),
        mean_fold_mae=(
            "mae",
            "mean"
        ),
        mean_fold_rmse=(
            "rmse",
            "mean"
        ),
        mean_fold_median_absolute_error=(
            "median_absolute_error",
            "mean"
        ),
    )
    .reset_index()
)


# ------------------------------------------------------------
# 16. Calculate pooled prediction-level metrics
# ------------------------------------------------------------
#
# Every held-out observation receives equal weight.
# ------------------------------------------------------------

pooled_results = []

for model_name in model_pipelines:

    model_predictions = (
        prediction_results_df[
            prediction_results_df[
                "model"
            ]
            == model_name
        ]
    )

    y_true = model_predictions[
        y_column
    ]

    y_pred = model_predictions[
        "prediction"
    ]

    pooled_results.append(
        {
            "model": model_name,

            "pooled_mae":
                mean_absolute_error(
                    y_true,
                    y_pred
                ),

            "pooled_rmse":
                np.sqrt(
                    mean_squared_error(
                        y_true,
                        y_pred
                    )
                ),

            "pooled_median_absolute_error":
                median_absolute_error(
                    y_true,
                    y_pred
                ),

            "pooled_records":
                len(model_predictions),
        }
    )


pooled_results_df = pd.DataFrame(
    pooled_results
)


# ------------------------------------------------------------
# 17. Combine model summaries
# ------------------------------------------------------------

model_comparison_df = (
    mean_fold_results
    .merge(
        pooled_results_df,
        on="model",
        how="inner"
    )
    .sort_values(
        "pooled_mae"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 18. Display fold-level results
# ------------------------------------------------------------

print(
    "\nFold-level model performance:"
)

display(
    fold_summary.round(3)
)


# ------------------------------------------------------------
# 19. Display overall model comparison
# ------------------------------------------------------------

print(
    "\nOverall model comparison:"
)

display(
    model_comparison_df.round(3)
)


# ------------------------------------------------------------
# 20. Identify the best initial ML model
# ------------------------------------------------------------

best_ml_model = (
    model_comparison_df.iloc[0][
        "model"
    ]
)

best_ml_mae = (
    model_comparison_df.iloc[0][
        "pooled_mae"
    ]
)

print(
    "\nBest initial ML model by pooled MAE:"
)

print(
    f"  {best_ml_model}"
)

print(
    f"  Pooled MAE: {best_ml_mae:.3f}"
)


# ------------------------------------------------------------
# 21. Final validation checks
# ------------------------------------------------------------

assert (
    fold_results_df[
        "fold"
    ].nunique()
    == 8
)

assert (
    prediction_results_df[
        "model"
    ].nunique()
    == 3
)

assert (
    prediction_results_df[
        "fold"
    ].nunique()
    == 8
)

assert (
    model_comparison_df[
        "pooled_records"
    ]
    .eq(expected_validation_records)
    .all()
)

assert (
    np.isfinite(
        model_comparison_df[
            [
                "pooled_mae",
                "pooled_rmse",
                "pooled_median_absolute_error",
            ]
        ]
    )
    .all()
    .all()
)


print(
    "\n" + "=" * 70
)
print(
    "ROLLING-ORIGIN MODEL EVALUATION COMPLETED"
)
print(
    "=" * 70
)

print(
    "\n✓ Eight chronological validation folds used"
)

print(
    "✓ Three regression models evaluated"
)

print(
    "✓ No random train/test split"
)

print(
    "✓ No global predictor imputation"
)

print(
    "✓ Preprocessing fitted within each training fold"
)

print(
    "✓ Future validation observations not used during fitting"
)

print(
    "✓ Unavailable outbreak targets were not treated as zero"
)

ROLLING-ORIGIN MODEL FITTING AND EVALUATION

Chronological modelling periods:


,year,semester_order,temporal_period,period_label
0,2020,1,4040,2020 Jan-Jun
1,2020,2,4041,2020 Jul-Dec
2,2021,1,4042,2021 Jan-Jun
3,2021,2,4043,2021 Jul-Dec
4,2022,1,4044,2022 Jan-Jun
5,2022,2,4045,2022 Jul-Dec
6,2023,1,4046,2023 Jan-Jun
7,2023,2,4047,2023 Jul-Dec
8,2024,1,4048,2024 Jan-Jun
9,2024,2,4049,2024 Jul-Dec



Fold-level model performance:


,model,fold,validation_period,validation_label,validation_year,validation_semester,training_records,validation_records,mae,rmse,median_absolute_error
0,Gradient Boosting,1,4044,2022 Jan-Jun,2022,1,143,45,6.062,15.110,2.933
1,Random Forest,1,4044,2022 Jan-Jun,2022,1,143,45,4.216,6.653,3.017
2,Ridge Regression,1,4044,2022 Jan-Jun,2022,1,143,45,12.778,31.542,7.223
3,Gradient Boosting,2,4045,2022 Jul-Dec,2022,2,188,49,8.471,23.158,2.192
4,Random Forest,2,4045,2022 Jul-Dec,2022,2,188,49,5.911,11.937,2.423
5,Ridge Regression,2,4045,2022 Jul-Dec,2022,2,188,49,8.190,22.897,3.288
6,Gradient Boosting,3,4046,2023 Jan-Jun,2023,1,237,22,8.196,12.892,5.170
7,Random Forest,3,4046,2023 Jan-Jun,2023,1,237,22,7.749,11.528,4.830
8,Ridge Regression,3,4046,2023 Jan-Jun,2023,1,237,22,16.070,42.390,6.495
9,Gradient Boosting,4,4047,2023 Jul-Dec,2023,2,259,22,8.110,14.439,2.722



Overall model comparison:


,model,validation_folds,evaluated_records,mean_fold_mae,mean_fold_rmse,mean_fold_median_absolute_error,pooled_mae,pooled_rmse,pooled_median_absolute_error,pooled_records
0,Random Forest,8,302,6.533,11.603,3.241,6.427,13.314,3.107,302
1,Gradient Boosting,8,302,7.215,16.230,2.696,7.342,19.572,2.435,302
2,Ridge Regression,8,302,8.707,18.595,4.514,8.051,20.725,4.220,302



Best initial ML model by pooled MAE:
  Random Forest
  Pooled MAE: 6.427

ROLLING-ORIGIN MODEL EVALUATION COMPLETED

✓ Eight chronological validation folds used
✓ Three regression models evaluated
✓ No random train/test split
✓ No global predictor imputation
✓ Preprocessing fitted within each training fold
✓ Future validation observations not used during fitting
✓ Unavailable outbreak targets were not treated as zero


In [18]:
# ============================================================
# STEP: Analyse rolling-origin model stability
# ============================================================

import pandas as pd
import numpy as np

# Recreate the fold-level evaluation results
# from the rolling-origin evaluation output.

fold_results = pd.DataFrame([
    ["Gradient Boosting", 1, 4044, "2022 Jan-Jun", 2022, 1, 143, 45, 6.062, 15.110, 2.933],
    ["Random Forest",     1, 4044, "2022 Jan-Jun", 2022, 1, 143, 45, 4.216,  6.653, 3.017],
    ["Ridge Regression",  1, 4044, "2022 Jan-Jun", 2022, 1, 143, 45, 12.778, 31.542, 7.223],

    ["Gradient Boosting", 2, 4045, "2022 Jul-Dec", 2022, 2, 188, 49, 8.471, 23.158, 2.192],
    ["Random Forest",     2, 4045, "2022 Jul-Dec", 2022, 2, 188, 49, 5.911, 11.937, 2.423],
    ["Ridge Regression",  2, 4045, "2022 Jul-Dec", 2022, 2, 188, 49, 8.190, 22.897, 3.288],

    ["Gradient Boosting", 3, 4046, "2023 Jan-Jun", 2023, 1, 237, 22, 8.196, 12.892, 5.170],
    ["Random Forest",     3, 4046, "2023 Jan-Jun", 2023, 1, 237, 22, 7.749, 11.528, 4.830],
    ["Ridge Regression",  3, 4046, "2023 Jan-Jun", 2023, 1, 237, 22, 16.070, 42.390, 6.495],

    ["Gradient Boosting", 4, 4047, "2023 Jul-Dec", 2023, 2, 259, 22, 8.110, 14.439, 2.722],
    ["Random Forest",     4, 4047, "2023 Jul-Dec", 2023, 2, 259, 22, 8.590, 13.754, 3.850],
    ["Ridge Regression",  4, 4047, "2023 Jul-Dec", 2023, 2, 259, 22, 11.156, 20.267, 5.199],

    ["Gradient Boosting", 5, 4048, "2024 Jan-Jun", 2024, 1, 281, 35, 4.094,  6.252, 2.020],
    ["Random Forest",     5, 4048, "2024 Jan-Jun", 2024, 1, 281, 35, 4.574,  6.597, 3.020],
    ["Ridge Regression",  5, 4048, "2024 Jan-Jun", 2024, 1, 281, 35, 3.787,  5.027, 2.831],

    ["Gradient Boosting", 6, 4049, "2024 Jul-Dec", 2024, 2, 316, 33, 7.231, 16.261, 2.845],
    ["Random Forest",     6, 4049, "2024 Jul-Dec", 2024, 2, 316, 33, 6.479,  9.189, 4.050],
    ["Ridge Regression",  6, 4049, "2024 Jul-Dec", 2024, 2, 316, 33, 6.479,  9.834, 4.370],

    ["Gradient Boosting", 7, 4050, "2025 Jan-Jun", 2025, 1, 349, 39, 5.434, 10.175, 1.914],
    ["Random Forest",     7, 4050, "2025 Jan-Jun", 2025, 1, 349, 39, 6.228, 10.745, 2.820],
    ["Ridge Regression",  7, 4050, "2025 Jan-Jun", 2025, 1, 349, 39, 7.143, 10.903, 4.276],

    ["Gradient Boosting", 8, 4051, "2025 Jul-Dec", 2025, 2, 388, 57, 10.120, 31.555, 1.775],
    ["Random Forest",     8, 4051, "2025 Jul-Dec", 2025, 2, 388, 57, 8.514, 22.424, 1.920],
    ["Ridge Regression",  8, 4051, "2025 Jul-Dec", 2025, 2, 388, 57, 4.055,  5.897, 2.433],
], columns=[
    "model",
    "fold",
    "validation_period",
    "validation_label",
    "validation_year",
    "validation_semester",
    "training_records",
    "validation_records",
    "mae",
    "rmse",
    "median_absolute_error",
])

# ------------------------------------------------------------
# 1. Model stability summary
# ------------------------------------------------------------

stability_summary = (
    fold_results
    .groupby("model")
    .agg(
        folds=("fold", "count"),
        mean_mae=("mae", "mean"),
        std_mae=("mae", "std"),
        min_mae=("mae", "min"),
        max_mae=("mae", "max"),
        mean_rmse=("rmse", "mean"),
        std_rmse=("rmse", "std"),
        mean_median_ae=("median_absolute_error", "mean"),
    )
    .round(3)
    .sort_values("mean_mae")
)

print("=" * 70)
print("MODEL STABILITY SUMMARY")
print("=" * 70)
display(stability_summary)


# ------------------------------------------------------------
# 2. Fold-by-fold MAE
# ------------------------------------------------------------

fold_mae = (
    fold_results
    .pivot(
        index="validation_label",
        columns="model",
        values="mae"
    )
    .sort_index()
    .round(3)
)

print("\n" + "=" * 70)
print("FOLD-BY-FOLD MAE")
print("=" * 70)
display(fold_mae)


# ------------------------------------------------------------
# 3. Fold-by-fold RMSE
# ------------------------------------------------------------

fold_rmse = (
    fold_results
    .pivot(
        index="validation_label",
        columns="model",
        values="rmse"
    )
    .sort_index()
    .round(3)
)

print("\n" + "=" * 70)
print("FOLD-BY-FOLD RMSE")
print("=" * 70)
display(fold_rmse)


# ------------------------------------------------------------
# 4. Best model in each fold
# ------------------------------------------------------------

best_by_fold = (
    fold_results
    .loc[
        fold_results.groupby("fold")["mae"].idxmin()
    ]
    [
        [
            "fold",
            "validation_label",
            "model",
            "mae",
            "rmse",
            "median_absolute_error",
        ]
    ]
    .sort_values("fold")
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("BEST MODEL BY VALIDATION FOLD")
print("=" * 70)
display(best_by_fold)


# ------------------------------------------------------------
# 5. Model fold wins
# ------------------------------------------------------------

fold_wins = (
    best_by_fold["model"]
    .value_counts()
    .rename_axis("model")
    .reset_index(name="fold_wins")
)

fold_wins["win_rate"] = (
    fold_wins["fold_wins"]
    / len(best_by_fold)
    * 100
).round(1)

print("\n" + "=" * 70)
print("MODEL FOLD WIN RATE")
print("=" * 70)
display(fold_wins)

MODEL STABILITY SUMMARY


,folds,mean_mae,std_mae,min_mae,max_mae,mean_rmse,std_rmse,mean_median_ae
model,,,,,,,,
Random Forest,8,6.533,1.661,4.216,8.59,11.603,5.042,3.241
Gradient Boosting,8,7.215,1.928,4.094,10.12,16.230,7.880,2.696
Ridge Regression,8,8.707,4.315,3.787,16.07,18.595,13.290,4.514



FOLD-BY-FOLD MAE


model,Gradient Boosting,Random Forest,Ridge Regression
validation_label,,,
2022 Jan-Jun,6.062,4.216,12.778
2022 Jul-Dec,8.471,5.911,8.190
2023 Jan-Jun,8.196,7.749,16.070
2023 Jul-Dec,8.110,8.590,11.156
2024 Jan-Jun,4.094,4.574,3.787
2024 Jul-Dec,7.231,6.479,6.479
2025 Jan-Jun,5.434,6.228,7.143
2025 Jul-Dec,10.120,8.514,4.055



FOLD-BY-FOLD RMSE


model,Gradient Boosting,Random Forest,Ridge Regression
validation_label,,,
2022 Jan-Jun,15.110,6.653,31.542
2022 Jul-Dec,23.158,11.937,22.897
2023 Jan-Jun,12.892,11.528,42.390
2023 Jul-Dec,14.439,13.754,20.267
2024 Jan-Jun,6.252,6.597,5.027
2024 Jul-Dec,16.261,9.189,9.834
2025 Jan-Jun,10.175,10.745,10.903
2025 Jul-Dec,31.555,22.424,5.897



BEST MODEL BY VALIDATION FOLD


,fold,validation_label,model,mae,rmse,median_absolute_error
0,1,2022 Jan-Jun,Random Forest,4.216,6.653,3.017
1,2,2022 Jul-Dec,Random Forest,5.911,11.937,2.423
2,3,2023 Jan-Jun,Random Forest,7.749,11.528,4.830
3,4,2023 Jul-Dec,Gradient Boosting,8.110,14.439,2.722
4,5,2024 Jan-Jun,Ridge Regression,3.787,5.027,2.831
5,6,2024 Jul-Dec,Random Forest,6.479,9.189,4.050
6,7,2025 Jan-Jun,Gradient Boosting,5.434,10.175,1.914
7,8,2025 Jul-Dec,Ridge Regression,4.055,5.897,2.433



MODEL FOLD WIN RATE


,model,fold_wins,win_rate
0,Random Forest,4,50.0
1,Gradient Boosting,2,25.0
2,Ridge Regression,2,25.0


In [19]:
# ============================================================
# STEP: Final Model Selection
# ============================================================

# Select the model with the best overall rolling-origin performance.
final_model_name = "Random Forest"

# Extract the performance summary for the selected model.
final_model_summary = stability_summary.loc[final_model_name]

print("=" * 70)
print("FINAL MODEL SELECTION")
print("=" * 70)

print(f"Selected model: {final_model_name}")
print(f"Validation folds: {int(final_model_summary['folds'])}")
print(f"Mean fold MAE: {final_model_summary['mean_mae']:.3f}")
print(f"MAE standard deviation: {final_model_summary['std_mae']:.3f}")
print(f"Best fold MAE: {final_model_summary['min_mae']:.3f}")
print(f"Worst fold MAE: {final_model_summary['max_mae']:.3f}")
print(f"Mean fold RMSE: {final_model_summary['mean_rmse']:.3f}")
print(f"RMSE standard deviation: {final_model_summary['std_rmse']:.3f}")
print(f"Mean median absolute error: {final_model_summary['mean_median_ae']:.3f}")

print("\nModel selection rationale:")
print(
    "Random Forest was selected because it achieved the lowest mean MAE "
    "and mean RMSE while also showing the lowest variability across the "
    "eight chronological validation folds."
)

print("\n" + "=" * 70)
print("FINAL MODEL CANDIDATE: RANDOM FOREST")
print("=" * 70)

FINAL MODEL SELECTION
Selected model: Random Forest
Validation folds: 8
Mean fold MAE: 6.533
MAE standard deviation: 1.661
Best fold MAE: 4.216
Worst fold MAE: 8.590
Mean fold RMSE: 11.603
RMSE standard deviation: 5.042
Mean median absolute error: 3.241

Model selection rationale:
Random Forest was selected because it achieved the lowest mean MAE and mean RMSE while also showing the lowest variability across the eight chronological validation folds.

FINAL MODEL CANDIDATE: RANDOM FOREST


In [20]:
# ============================================================
# STEP: Inspect Current Notebook Variables
# ============================================================

print("Variables currently available:")
print("=" * 70)

for name in sorted([
    name for name in globals()
    if not name.startswith("_")
]):
    obj = globals()[name]

    if hasattr(obj, "shape"):
        print(f"{name:<35} {type(obj).__name__:<20} shape={obj.shape}")
    else:
        print(f"{name:<35} {type(obj).__name__}")

print("=" * 70)

Variables currently available:
ColumnTransformer                   ABCMeta
DATA_PATH                           WindowsPath
GradientBoostingRegressor           ABCMeta
In                                  list
MIN_TRAIN_PERIODS                   int
MODELLING_FEATURES_PATH             WindowsPath
Out                                 dict
PROJECT_ROOT                        WindowsPath
Path                                type
Pipeline                            ABCMeta
RandomForestRegressor               ABCMeta
Ridge                               ABCMeta
SRC_PATH                            WindowsPath
SimpleImputer                       type
StandardScaler                      type
TEST_PERIODS                        int
X_columns                           list
X_train                             DataFrame            shape=(388, 27)
X_validation                        DataFrame            shape=(57, 27)
actual_prediction_records           int
actual_semesters                    list
actua

In [21]:
# ============================================================
# STEP 3: FINAL RANDOM FOREST TRAINING
# ============================================================

from sklearn.base import clone

print("=" * 70)
print("FINAL RANDOM FOREST TRAINING")
print("=" * 70)

# ------------------------------------------------------------
# 1. Prepare the complete eligible modelling dataset
# ------------------------------------------------------------

final_training_df = model_df.copy()

# Use the same predictors and target used during model evaluation.
X_final = final_training_df[X_columns].copy()
y_final = final_training_df[y_column].copy()

print(f"Eligible training records: {len(final_training_df)}")
print(f"Number of predictors: {X_final.shape[1]}")
print(f"Target column: {y_column}")

# ------------------------------------------------------------
# 2. Clone the validated Random Forest pipeline
# ------------------------------------------------------------

final_random_forest_pipeline = clone(random_forest_pipeline)

# ------------------------------------------------------------
# 3. Fit the pipeline on ALL eligible historical data
# ------------------------------------------------------------

final_random_forest_pipeline.fit(X_final, y_final)

print("\nFinal model training completed successfully.")

# ------------------------------------------------------------
# 4. Confirm fitted pipeline structure
# ------------------------------------------------------------

print("\nPipeline:")
print(final_random_forest_pipeline)

print("\n" + "=" * 70)
print("FINAL MODEL TRAINED")
print("=" * 70)
print("Model: Random Forest")
print(f"Training records: {len(X_final)}")
print(f"Features: {X_final.shape[1]}")
print("Preprocessing: fitted within final training pipeline")
print("Training data: all eligible historical observations")
print("=" * 70)

FINAL RANDOM FOREST TRAINING
Eligible training records: 445
Number of predictors: 27
Target column: target_outbreak_count

Final model training completed successfully.

Pipeline:
Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('numeric',
                                                  SimpleImputer(strategy='median'),
                                                  ['year', 'semester_order',
                                                   'is_first_semester',
                                                   'is_second_semester',
                                                   'semester_sin',
                                                   'semester_cos',
                                                   'years_since_2019',
                                                   'year_sin', 'year_cos',
                                                   'has_adm1',
                                                   'is_country_level',
         

In [22]:
# ============================================================
# STEP 4: FINAL MODEL SANITY CHECK
# ============================================================

print("=" * 70)
print("FINAL RANDOM FOREST — SANITY CHECK")
print("=" * 70)

# ------------------------------------------------------------
# 1. Generate predictions on the complete training data
# ------------------------------------------------------------

final_predictions = final_random_forest_pipeline.predict(X_final)

print(f"Prediction records: {len(final_predictions)}")
print(f"Expected records:   {len(y_final)}")

# ------------------------------------------------------------
# 2. Validate prediction integrity
# ------------------------------------------------------------

prediction_count_match = len(final_predictions) == len(y_final)
no_nan_predictions = np.isfinite(final_predictions).all()
non_negative_predictions = (final_predictions >= 0).all()

print("\nPrediction integrity:")
print(f"Record count matches target: {prediction_count_match}")
print(f"No NaN/infinite predictions: {no_nan_predictions}")
print(f"All predictions >= 0:        {non_negative_predictions}")

# ------------------------------------------------------------
# 3. Calculate training-set diagnostics
# ------------------------------------------------------------

final_train_mae = mean_absolute_error(y_final, final_predictions)
final_train_rmse = np.sqrt(
    mean_squared_error(y_final, final_predictions)
)
final_train_median_ae = median_absolute_error(
    y_final, final_predictions
)

print("\nTraining-set diagnostics:")
print(f"MAE:                 {final_train_mae:.3f}")
print(f"RMSE:                {final_train_rmse:.3f}")
print(f"Median absolute AE:  {final_train_median_ae:.3f}")

# ------------------------------------------------------------
# 4. Prediction range
# ------------------------------------------------------------

print("\nPrediction range:")
print(f"Minimum prediction: {final_predictions.min():.3f}")
print(f"Maximum prediction: {final_predictions.max():.3f}")
print(f"Mean prediction:    {final_predictions.mean():.3f}")

# ------------------------------------------------------------
# 5. Final validation status
# ------------------------------------------------------------

sanity_check_passed = (
    prediction_count_match
    and no_nan_predictions
    and non_negative_predictions
)

print("\n" + "=" * 70)

if sanity_check_passed:
    print("FINAL MODEL SANITY CHECK: PASSED")
else:
    print("FINAL MODEL SANITY CHECK: FAILED")

print("=" * 70)

FINAL RANDOM FOREST — SANITY CHECK
Prediction records: 445
Expected records:   445

Prediction integrity:
Record count matches target: True
No NaN/infinite predictions: True
All predictions >= 0:        True

Training-set diagnostics:
MAE:                 1.483
RMSE:                3.579
Median absolute AE:  0.683

Prediction range:
Minimum prediction: 1.047
Maximum prediction: 130.163
Mean prediction:    6.850

FINAL MODEL SANITY CHECK: PASSED


In [24]:
# ============================================================
# STEP 5: SAVE FINAL MODEL ARTEFACT — CORRECTED
# ============================================================

import json
import joblib
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. Define artefact directory
# ------------------------------------------------------------

ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "final_model"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Save the complete fitted pipeline
# ------------------------------------------------------------

MODEL_ARTIFACT_PATH = ARTIFACT_DIR / "random_forest_pipeline.joblib"

joblib.dump(
    final_random_forest_pipeline,
    MODEL_ARTIFACT_PATH
)

# ------------------------------------------------------------
# 3. Determine training period from the existing temporal data
# ------------------------------------------------------------

training_period_start = int(temporal_values.min())
training_period_end = int(temporal_values.max())

# ------------------------------------------------------------
# 4. Create model metadata
# ------------------------------------------------------------

model_metadata = {
    "model_name": "Random Forest",
    "model_type": "RandomForestRegressor",
    "model_version": "1.0.0",

    "target_column": y_column,

    "n_training_records": int(len(X_final)),
    "n_features": int(X_final.shape[1]),
    "features": list(X_columns),

    "training_period": {
        "start_period": training_period_start,
        "end_period": training_period_end
    },

    "validation": {
        "method": "rolling_origin_temporal_validation",
        "folds": 8,
        "mean_fold_mae": float(final_model_summary["mean_mae"]),
        "std_fold_mae": float(final_model_summary["std_mae"]),
        "mean_fold_rmse": float(final_model_summary["mean_rmse"]),
        "std_fold_rmse": float(final_model_summary["std_rmse"]),
        "mean_fold_median_absolute_error": float(
            final_model_summary["mean_median_ae"]
        )
    },

    "training_diagnostics": {
        "mae": float(final_train_mae),
        "rmse": float(final_train_rmse),
        "median_absolute_error": float(final_train_median_ae)
    },

    "prediction_sanity_check": {
        "prediction_records": int(len(final_predictions)),
        "no_nan_or_infinite_predictions": bool(no_nan_predictions),
        "non_negative_predictions": bool(non_negative_predictions),
        "minimum_prediction": float(final_predictions.min()),
        "maximum_prediction": float(final_predictions.max()),
        "mean_prediction": float(final_predictions.mean())
    },

    "random_forest_parameters": {
        "n_estimators": 300,
        "random_state": 42,
        "n_jobs": -1
    },

    "created_at_utc": datetime.now(timezone.utc).isoformat()
}

# ------------------------------------------------------------
# 5. Save metadata
# ------------------------------------------------------------

METADATA_PATH = ARTIFACT_DIR / "model_metadata.json"

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(model_metadata, f, indent=2)

# ------------------------------------------------------------
# 6. Verify artefacts
# ------------------------------------------------------------

print("=" * 70)
print("FINAL MODEL ARTEFACTS")
print("=" * 70)

print(f"Model artefact:   {MODEL_ARTIFACT_PATH}")
print(f"Metadata file:    {METADATA_PATH}")

print(f"\nModel file exists:    {MODEL_ARTIFACT_PATH.exists()}")
print(f"Metadata file exists: {METADATA_PATH.exists()}")

print("\nTraining period:")
print(f"Start temporal period: {training_period_start}")
print(f"End temporal period:   {training_period_end}")

print("\nModel summary:")
print(f"Model:              {model_metadata['model_name']}")
print(f"Training records:   {model_metadata['n_training_records']}")
print(f"Features:           {model_metadata['n_features']}")
print(f"Validation folds:   {model_metadata['validation']['folds']}")
print(f"Rolling MAE:        {model_metadata['validation']['mean_fold_mae']:.3f}")
print(f"Rolling RMSE:       {model_metadata['validation']['mean_fold_rmse']:.3f}")

print("\n" + "=" * 70)
print("MODEL ARTEFACT CREATION COMPLETE")
print("=" * 70)

FINAL MODEL ARTEFACTS
Model artefact:   c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria\artifacts\final_model\random_forest_pipeline.joblib
Metadata file:    c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria\artifacts\final_model\model_metadata.json

Model file exists:    True
Metadata file exists: True

Training period:
Start temporal period: 4041
End temporal period:   4052

Model summary:
Model:              Random Forest
Training records:   445
Features:           27
Validation folds:   8
Rolling MAE:        6.533
Rolling RMSE:       11.603

MODEL ARTEFACT CREATION COMPLETE


In [25]:
# ============================================================
# STEP 6: VERIFY FINAL TRAINING PERIOD
# ============================================================

print("=" * 70)
print("FINAL TRAINING PERIOD VERIFICATION")
print("=" * 70)

print("\nmodel_df:")
print(f"Rows: {len(model_df)}")

print("\nTemporal values:")
print(f"Minimum: {temporal_values.min()}")
print(f"Maximum: {temporal_values.max()}")

print("\nRelevant temporal columns in model_df:")

temporal_columns = [
    col for col in model_df.columns
    if "period" in col.lower()
    or "year" in col.lower()
    or "semester" in col.lower()
]

print(temporal_columns)

if temporal_columns:
    print("\nUnique temporal values:")
    for col in temporal_columns:
        print(f"\n{col}:")
        print(model_df[col].unique()[:20])

print("\n" + "=" * 70)

FINAL TRAINING PERIOD VERIFICATION

model_df:
Rows: 445

Temporal values:
Minimum: 4041
Maximum: 4052

Relevant temporal columns in model_df:
['year', 'semester_order', 'is_first_semester', 'is_second_semester', 'semester_sin', 'semester_cos', 'years_since_2019', 'year_sin', 'year_cos', 'previous_semester_outbreak_count', 'two_semesters_prior_outbreak_count', 'semester_total_precipitation_mm', 'semester_mean_wind_component_magnitude_ms']

Unique temporal values:

year:
[2021 2025 2022 2020 2023 2024]

semester_order:
[2 1]

is_first_semester:
[0 1]

is_second_semester:
[1 0]

semester_sin:
[1.2246468e-16 0.0000000e+00]

semester_cos:
[-1.  1.]

years_since_2019:
[2 6 3 1 4 5]

year_sin:
[-9.51056516e-01 -2.44929360e-16 -8.09016994e-01 -1.00000000e+00
 -5.87785252e-01 -3.09016994e-01]

year_cos:
[ 3.09016994e-01  1.00000000e+00  5.87785252e-01 -1.83697020e-16
  8.09016994e-01  9.51056516e-01]

previous_semester_outbreak_count:
[ nan   7.   2.   5.   1.   4.   9.   8.   3.   6.  48.  39.

In [26]:
# ============================================================
# STEP 7: MODEL ARTEFACT RELOAD & REPRODUCIBILITY TEST
# ============================================================

import joblib
import numpy as np

print("=" * 70)
print("MODEL ARTEFACT RELOAD & REPRODUCIBILITY TEST")
print("=" * 70)

# ------------------------------------------------------------
# 1. Load the saved pipeline from disk
# ------------------------------------------------------------

reloaded_pipeline = joblib.load(MODEL_ARTIFACT_PATH)

print("\nSaved pipeline successfully reloaded.")

# ------------------------------------------------------------
# 2. Generate predictions using the reloaded model
# ------------------------------------------------------------

reloaded_predictions = reloaded_pipeline.predict(X_final)

print(f"Reloaded prediction records: {len(reloaded_predictions)}")

# ------------------------------------------------------------
# 3. Compare predictions against the original model
# ------------------------------------------------------------

predictions_identical = np.array_equal(
    final_predictions,
    reloaded_predictions
)

max_prediction_difference = np.max(
    np.abs(final_predictions - reloaded_predictions)
)

mean_prediction_difference = np.mean(
    np.abs(final_predictions - reloaded_predictions)
)

print("\nPrediction reproducibility:")
print(f"Predictions exactly identical: {predictions_identical}")
print(f"Maximum prediction difference: {max_prediction_difference:.12f}")
print(f"Mean prediction difference:    {mean_prediction_difference:.12f}")

# ------------------------------------------------------------
# 4. Validate reloaded predictions
# ------------------------------------------------------------

reloaded_no_nan = np.isfinite(reloaded_predictions).all()
reloaded_non_negative = (reloaded_predictions >= 0).all()

print("\nReloaded model integrity:")
print(f"No NaN/infinite predictions: {reloaded_no_nan}")
print(f"All predictions >= 0:        {reloaded_non_negative}")

# ------------------------------------------------------------
# 5. Calculate metrics from reloaded model
# ------------------------------------------------------------

reloaded_mae = mean_absolute_error(
    y_final,
    reloaded_predictions
)

reloaded_rmse = np.sqrt(
    mean_squared_error(
        y_final,
        reloaded_predictions
    )
)

reloaded_median_ae = median_absolute_error(
    y_final,
    reloaded_predictions
)

print("\nReloaded model diagnostics:")
print(f"MAE:                 {reloaded_mae:.3f}")
print(f"RMSE:                {reloaded_rmse:.3f}")
print(f"Median absolute AE:  {reloaded_median_ae:.3f}")

# ------------------------------------------------------------
# 6. Final reproducibility decision
# ------------------------------------------------------------

reproducibility_passed = (
    predictions_identical
    and reloaded_no_nan
    and reloaded_non_negative
)

print("\n" + "=" * 70)

if reproducibility_passed:
    print("ARTEFACT REPRODUCIBILITY TEST: PASSED")
else:
    print("ARTEFACT REPRODUCIBILITY TEST: FAILED")

print("=" * 70)

MODEL ARTEFACT RELOAD & REPRODUCIBILITY TEST

Saved pipeline successfully reloaded.
Reloaded prediction records: 445

Prediction reproducibility:
Predictions exactly identical: True
Maximum prediction difference: 0.000000000000
Mean prediction difference:    0.000000000000

Reloaded model integrity:
No NaN/infinite predictions: True
All predictions >= 0:        True

Reloaded model diagnostics:
MAE:                 1.483
RMSE:                3.579
Median absolute AE:  0.683

ARTEFACT REPRODUCIBILITY TEST: PASSED


In [27]:
# ============================================================
# STEP 8: PRODUCTION INFERENCE FUNCTION
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Define production inference function
# ------------------------------------------------------------

def predict_outbreak_count(input_data, model=reloaded_pipeline):
    """
    Generate outbreak-count predictions using the saved
    Random Forest production pipeline.

    Parameters
    ----------
    input_data : pandas.DataFrame
        DataFrame containing the 27 production model features.

    model : sklearn Pipeline
        Fitted preprocessing + Random Forest pipeline.

    Returns
    -------
    pandas.DataFrame
        Prediction results with one prediction per input record.
    """

    # --------------------------------------------------------
    # Validate input type
    # --------------------------------------------------------

    if not isinstance(input_data, pd.DataFrame):
        raise TypeError(
            "input_data must be a pandas DataFrame."
        )

    # --------------------------------------------------------
    # Validate required features
    # --------------------------------------------------------

    missing_features = [
        feature
        for feature in X_columns
        if feature not in input_data.columns
    ]

    if missing_features:
        raise ValueError(
            "Missing required model features: "
            + ", ".join(missing_features)
        )

    # --------------------------------------------------------
    # Select features in the exact training order
    # --------------------------------------------------------

    X_input = input_data[X_columns].copy()

    # --------------------------------------------------------
    # Generate predictions
    # --------------------------------------------------------

    predictions = model.predict(X_input)

    # --------------------------------------------------------
    # Validate predictions
    # --------------------------------------------------------

    if not np.isfinite(predictions).all():
        raise ValueError(
            "Model generated NaN or infinite predictions."
        )

    if (predictions < 0).any():
        raise ValueError(
            "Model generated negative outbreak-count predictions."
        )

    # --------------------------------------------------------
    # Return clean prediction output
    # --------------------------------------------------------

    prediction_output = pd.DataFrame({
        "predicted_outbreak_count": predictions
    })

    return prediction_output


# ============================================================
# STEP 8A: TEST THE PRODUCTION INFERENCE FUNCTION
# ============================================================

inference_test = predict_outbreak_count(X_final)

print("=" * 70)
print("PRODUCTION INFERENCE TEST")
print("=" * 70)

print(f"Input records:       {len(X_final)}")
print(f"Prediction records:  {len(inference_test)}")

print("\nPrediction output:")
print(inference_test.head())

print("\nPrediction summary:")
print(
    inference_test["predicted_outbreak_count"].describe()
)

print("\n" + "=" * 70)
print("PRODUCTION INFERENCE FUNCTION TEST: PASSED")
print("=" * 70)

PRODUCTION INFERENCE TEST
Input records:       445
Prediction records:  445

Prediction output:
   predicted_outbreak_count
0                  5.660000
1                  2.423333
2                  1.996667
3                  4.926667
4                  2.683333

Prediction summary:
count    445.000000
mean       6.850195
std       10.458543
min        1.046667
25%        2.056667
50%        3.410000
75%        7.353333
max      130.163333
Name: predicted_outbreak_count, dtype: float64

PRODUCTION INFERENCE FUNCTION TEST: PASSED


In [28]:
# ============================================================
# STEP 9: PRODUCTION FEATURE SCHEMA
# ============================================================

import json

print("=" * 70)
print("PRODUCTION FEATURE SCHEMA")
print("=" * 70)

# ------------------------------------------------------------
# 1. Extract fitted preprocessing information
# ------------------------------------------------------------

fitted_preprocessor = reloaded_pipeline.named_steps["preprocess"]
fitted_model = reloaded_pipeline.named_steps["model"]

# ------------------------------------------------------------
# 2. Create production schema
# ------------------------------------------------------------

production_schema = {
    "schema_version": "1.0.0",

    "model": {
        "name": "Random Forest",
        "version": model_metadata["model_version"],
        "type": type(fitted_model).__name__
    },

    "target": {
        "name": y_column,
        "type": "numeric",
        "description": "Predicted outbreak count"
    },

    "features": {
        "count": len(X_columns),
        "ordered_names": list(X_columns)
    },

    "training": {
        "records": model_metadata["n_training_records"],
        "start_temporal_period": model_metadata[
            "training_period"
        ]["start_period"],
        "end_temporal_period": model_metadata[
            "training_period"
        ]["end_period"]
    },

    "validation": {
        "method": model_metadata["validation"]["method"],
        "folds": model_metadata["validation"]["folds"],
        "mean_mae": model_metadata["validation"]["mean_fold_mae"],
        "mean_rmse": model_metadata["validation"]["mean_fold_rmse"]
    },

    "input_requirements": {
        "required_columns": list(X_columns),
        "column_order_required": True,
        "missing_columns_allowed": False,
        "extra_columns_allowed": True
    }
}

# ------------------------------------------------------------
# 3. Save schema
# ------------------------------------------------------------

SCHEMA_PATH = ARTIFACT_DIR / "production_schema.json"

with open(SCHEMA_PATH, "w", encoding="utf-8") as f:
    json.dump(production_schema, f, indent=2)

# ------------------------------------------------------------
# 4. Validate schema against actual model
# ------------------------------------------------------------

schema_features = production_schema["features"]["ordered_names"]

schema_matches_training_features = (
    schema_features == list(X_columns)
)

schema_feature_count_matches = (
    production_schema["features"]["count"]
    == X_final.shape[1]
)

pipeline_feature_count_matches = (
    len(X_columns) == len(
        fitted_preprocessor.transformers_[0][2]
    )
)

# ------------------------------------------------------------
# 5. Print validation results
# ------------------------------------------------------------

print("\nSchema file:")
print(SCHEMA_PATH)

print(f"\nSchema file exists: {SCHEMA_PATH.exists()}")

print("\nSchema validation:")
print(
    f"Feature names match training schema: "
    f"{schema_matches_training_features}"
)

print(
    f"Feature count matches training data: "
    f"{schema_feature_count_matches}"
)

print(
    f"Pipeline feature count matches: "
    f"{pipeline_feature_count_matches}"
)

print(f"\nNumber of production features: {len(X_columns)}")

print("\nOrdered production features:")

for i, feature_name in enumerate(X_columns, start=1):
    print(f"{i:02d}. {feature_name}")

# ------------------------------------------------------------
# 6. Final schema decision
# ------------------------------------------------------------

schema_validation_passed = (
    schema_matches_training_features
    and schema_feature_count_matches
    and pipeline_feature_count_matches
)

print("\n" + "=" * 70)

if schema_validation_passed:
    print("PRODUCTION FEATURE SCHEMA: PASSED")
else:
    print("PRODUCTION FEATURE SCHEMA: FAILED")

print("=" * 70)

PRODUCTION FEATURE SCHEMA

Schema file:
c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria\artifacts\final_model\production_schema.json

Schema file exists: True

Schema validation:
Feature names match training schema: True
Feature count matches training data: True
Pipeline feature count matches: True

Number of production features: 27

Ordered production features:
01. year
02. semester_order
03. is_first_semester
04. is_second_semester
05. semester_sin
06. semester_cos
07. years_since_2019
08. year_sin
09. year_cos
10. has_adm1
11. is_country_level
12. disease_code
13. previous_semester_outbreak_count
14. two_semesters_prior_outbreak_count
15. outbreak_history_mean_3
16. outbreak_history_max_3
17. climate_available
18. temperature_missing
19. dewpoint_missing
20. precipitation_missing
21. wind_missing
22. climate_variable_count
23. climate_complete
24. temperature_c
25. dewpoint_c
26. semester_total_precipitation_mm
27. semester_mean_wind_component_magnitude_ms

PRODUCT

In [29]:
# ============================================================
# STEP 10: SINGLE-RECORD PRODUCTION INFERENCE TEST
# ============================================================

print("=" * 70)
print("SINGLE-RECORD PRODUCTION INFERENCE TEST")
print("=" * 70)

# ------------------------------------------------------------
# 1. Select one historical observation
# ------------------------------------------------------------

single_record = X_final.iloc[[0]].copy()

print("\nInput record:")
print(single_record.T)

# ------------------------------------------------------------
# 2. Run production inference
# ------------------------------------------------------------

single_prediction = predict_outbreak_count(
    single_record
)

# ------------------------------------------------------------
# 3. Extract prediction
# ------------------------------------------------------------

predicted_count = float(
    single_prediction["predicted_outbreak_count"].iloc[0]
)

print("\nPrediction:")
print(f"Predicted outbreak count: {predicted_count:.3f}")

# ------------------------------------------------------------
# 4. Validate result
# ------------------------------------------------------------

single_prediction_valid = (
    len(single_prediction) == 1
    and np.isfinite(predicted_count)
    and predicted_count >= 0
)

print("\nPrediction validation:")
print(f"Exactly one prediction returned: {len(single_prediction) == 1}")
print(f"Prediction is finite:            {np.isfinite(predicted_count)}")
print(f"Prediction is non-negative:      {predicted_count >= 0}")

# ------------------------------------------------------------
# 5. Final decision
# ------------------------------------------------------------

print("\n" + "=" * 70)

if single_prediction_valid:
    print("SINGLE-RECORD INFERENCE TEST: PASSED")
else:
    print("SINGLE-RECORD INFERENCE TEST: FAILED")

print("=" * 70)

SINGLE-RECORD PRODUCTION INFERENCE TEST

Input record:
                                                      1
year                                       2.021000e+03
semester_order                             2.000000e+00
is_first_semester                          0.000000e+00
is_second_semester                         1.000000e+00
semester_sin                               1.224647e-16
semester_cos                              -1.000000e+00
years_since_2019                           2.000000e+00
year_sin                                  -9.510565e-01
year_cos                                   3.090170e-01
has_adm1                                   1.000000e+00
is_country_level                           0.000000e+00
disease_code                               4.000000e+00
previous_semester_outbreak_count                    NaN
two_semesters_prior_outbreak_count                  NaN
outbreak_history_mean_3                             NaN
outbreak_history_max_3                           

In [31]:
# ==============================================================
# SAVE FINAL EVALUATION DATASET
# ==============================================================
# The target variable "target_outbreak_count" was created during
# the modelling preparation in Notebook 08 and is not stored in
# the processed CSV files.
#
# To ensure Notebook 09 evaluates exactly the same observations
# used during model development, we save a frozen copy of the
# final modelling dataset.
#
# This dataset contains:
#   - 27 model predictor variables
#   - 1 target variable
#   - 445 eligible historical observations
#
# We also save the chronological period values used by the
# rolling-origin temporal validation.
# ==============================================================

from pathlib import Path
import pandas as pd

# --------------------------------------------------------------
# 1. Define the project root explicitly
# --------------------------------------------------------------
# Notebook 08 is located inside the "notebooks" directory.
# Therefore, its parent directory is the project root.
# --------------------------------------------------------------

project_root = Path.cwd().parent

print(f"Project root: {project_root}")

# --------------------------------------------------------------
# 2. Create the evaluation artefact directory
# --------------------------------------------------------------

evaluation_artifacts_directory = (
    project_root / "artifacts" / "evaluation"
)

evaluation_artifacts_directory.mkdir(
    parents=True,
    exist_ok=True
)

# --------------------------------------------------------------
# 3. Define output paths
# --------------------------------------------------------------

evaluation_dataset_path = (
    evaluation_artifacts_directory
    / "model_evaluation_dataset.csv"
)

temporal_values_path = (
    evaluation_artifacts_directory
    / "temporal_period_values.csv"
)

# --------------------------------------------------------------
# 4. Save the exact modelling dataset
# --------------------------------------------------------------
# model_df is the 445-row dataset used for final model training.
# It contains the 27 predictors plus the target.
# --------------------------------------------------------------

model_df.to_csv(
    evaluation_dataset_path,
    index=False
)

# --------------------------------------------------------------
# 5. Save the temporal period information
# --------------------------------------------------------------
# temporal_values contains the chronological period associated
# with each modelling observation and is required to reproduce
# the rolling-origin evaluation.
# --------------------------------------------------------------

pd.DataFrame({
    "temporal_period": temporal_values
}).to_csv(
    temporal_values_path,
    index=False
)

# --------------------------------------------------------------
# 6. Validate the saved evaluation dataset
# --------------------------------------------------------------

expected_evaluation_columns = X_columns + [y_column]

columns_match = (
    model_df.columns.tolist() == expected_evaluation_columns
)

print("\n" + "=" * 70)
print("EVALUATION DATASET SAVED")
print("=" * 70)

print("\nEvaluation dataset:")
print(f"  Path:   {evaluation_dataset_path}")
print(f"  Exists: {evaluation_dataset_path.exists()}")
print(f"  Shape:  {model_df.shape}")

print("\nTemporal period values:")
print(f"  Path:   {temporal_values_path}")
print(f"  Exists: {temporal_values_path.exists()}")
print(f"  Records: {len(temporal_values)}")

print("\nValidation:")
print(f"  Expected records: 445")
print(f"  Actual records:   {len(model_df)}")
print(f"  Expected columns: 28")
print(f"  Actual columns:   {model_df.shape[1]}")
print(f"  Column order valid: {columns_match}")

if (
    len(model_df) == 445
    and model_df.shape[1] == 28
    and columns_match
    and len(temporal_values) == 445
):
    print("\n" + "=" * 70)
    print("EVALUATION DATASET VALIDATION: PASSED")
    print("=" * 70)
else:
    raise ValueError(
        "Evaluation dataset validation failed."
    )

Project root: c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria

EVALUATION DATASET SAVED

Evaluation dataset:
  Path:   c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria\artifacts\evaluation\model_evaluation_dataset.csv
  Exists: True
  Shape:  (445, 28)

Temporal period values:
  Path:   c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria\artifacts\evaluation\temporal_period_values.csv
  Exists: True
  Records: 445

Validation:
  Expected records: 445
  Actual records:   445
  Expected columns: 28
  Actual columns:   28
  Column order valid: True

EVALUATION DATASET VALIDATION: PASSED
